# Sourav's Wedding Gallery - Enhanced Face Processor

This notebook uses deep learning to detect and group faces in your wedding gallery. 

### **New Features in this version:**
- **CNN Model Support**: Uses GPU acceleration for much better face detection.
- **Upsampling**: Detects even small faces in large group photos.
- **Improved Clustering**: Better separation of different people.
- **Progress Tracking**: See exactly what's happening during processing.

---

### Step 1: Set up GPU and Install Dependencies
Go to **Runtime** > **Change runtime type** and ensure **T4 GPU** is selected for best performance.

In [ ]:
!pip install face_recognition requests numpy Pillow scikit-learn

### Step 2: Upload `manifest.json`

In [ ]:
from google.colab import files
import os

uploaded = files.upload()

if 'manifest.json' not in uploaded:
    print("❌ Please upload 'manifest.json'!")
else:
    print("✅ manifest.json uploaded successfully.")

### Step 3: Run Advanced Image Processing
We will use the **CNN model** which is highly accurate at finding faces.

In [ ]:
import json
import requests
import numpy as np
from PIL import Image
import face_recognition
from sklearn.cluster import DBSCAN
import shutil
import torch

# Settings
DETECTION_MODEL = 'cnn' if torch.cuda.is_available() else 'hog'
UPSAMPLE_TIMES = 1
DISTANCE_THRESHOLD = 0.5 # Lower = more strict, Higher = more relaxed

MANIFEST_PATH = "manifest.json"
OUTPUT_FACES_PATH = "faces.json"
THUMBNAILS_DIR = "face-thumbnails"
IMAGES_DIR = "downloaded_images"

if DETECTION_MODEL == 'cnn':
    print("🚀 GPU detected! Using high-accuracy CNN model.")
else:
    print("⚠️ GPU not detected. Using HOG model (faster but less accurate).")

if not os.path.exists(THUMBNAILS_DIR):
    os.makedirs(THUMBNAILS_DIR)
if not os.path.exists(IMAGES_DIR):
    os.makedirs(IMAGES_DIR)

with open(MANIFEST_PATH, 'r') as f:
    manifest = json.load(f)

known_encodings = []
image_data = []

print(f"Processing {len(manifest)} images...")

for i, item in enumerate(manifest):
    public_id = item['public_id']
    url = item['url']
    filename = public_id.replace('/', '_') + ".jpg"
    filepath = os.path.join(IMAGES_DIR, filename)

    if not os.path.exists(filepath):
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                with open(filepath, 'wb') as f:
                    f.write(r.content)
        except:
            continue

    if os.path.exists(filepath):
        try:
            image = face_recognition.load_image_file(filepath)
            # Detect faces
            face_locations = face_recognition.face_locations(image, number_of_times_to_upsample=UPSAMPLE_TIMES, model=DETECTION_MODEL)
            face_encodings = face_recognition.face_encodings(image, face_locations)

            if i % 10 == 0:
                print(f"Progress: {i}/{len(manifest)} images...")

            for loc, enc in zip(face_locations, face_encodings):
                known_encodings.append(enc)
                image_data.append({"public_id": public_id, "location": loc, "filepath": filepath})
        except Exception as e:
            print(f"  - Error in {public_id}: {e}")

if known_encodings:
    print(f"\nDetected {len(known_encodings)} total faces. Grouping them into people...")
    
    # DBSCAN Clustering
    clt = DBSCAN(metric="euclidean", eps=DISTANCE_THRESHOLD, min_samples=3, n_jobs=-1)
    clt.fit(known_encodings)
    label_ids = np.unique(clt.labels_)

    faces_result = []
    for label_id in label_ids:
        if label_id == -1: continue
        indexes = np.where(clt.labels_ == label_id)[0]
        rep_face = image_data[indexes[0]]
        
        img = Image.open(rep_face['filepath'])
        t, r, b, l = rep_face['location']
        w, h = img.size
        pad_h, pad_w = (b-t)//2, (r-l)//2
        face_img = img.crop((max(0, l-pad_w), max(0, t-pad_h), min(w, r+pad_w), min(h, b+pad_h)))
        face_img.thumbnail((200, 200))
        
        thumb_name = f"face_{label_id}.jpg"
        face_img.save(os.path.join(THUMBNAILS_DIR, thumb_name), "JPEG", quality=90)
        
        photos = sorted(list(set([image_data[idx]['public_id'] for idx in indexes])))
        faces_result.append({
            "id": int(label_id), 
            "name": f"Person {label_id+1}", 
            "thumbnail": f"/face-thumbnails/{thumb_name}", 
            "photos": photos,
            "count": len(photos)
        })

    # Sort by photo count
    faces_result.sort(key=lambda x: x['count'], reverse=True)
    
    with open(OUTPUT_FACES_PATH, 'w') as f:
        json.dump(faces_result, f, indent=2)
    
    print(f"\n✅ Found {len(faces_result)} unique people!")
    print(f"✅ Results saved to {OUTPUT_FACES_PATH}")
else:
    print("❌ No faces found. Try increasing UPSAMPLE_TIMES to 2.")

### Step 4: Download Results

In [ ]:
import shutil
from google.colab import files

if os.path.exists(OUTPUT_FACES_PATH):
    os.makedirs("results", exist_ok=True)
    shutil.copy(OUTPUT_FACES_PATH, "results/faces.json")
    if os.path.exists("results/face-thumbnails"):
        shutil.rmtree("results/face-thumbnails")
    shutil.copytree(THUMBNAILS_DIR, "results/face-thumbnails")
    
    shutil.make_archive("gallery_faces", 'zip', "results")
    files.download("gallery_faces.zip")
    print("✅ Zip file created and download started!")
else:
    print("❌ Nothing to download.")